# CEMS Data Ingestion

## Executive summary
Read Copernicus CEMS vectors from `input/copernicus/{activation_id}`. No Copernicus API call is made.

## Prerequisites
Install `requirements.txt` and place `data.gpkg` or `data.geojson` in the activation folder. The activation ID is read from `config/params.yaml`.

## Parameters

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.copernicus_loader import copernicus_file, load_params

params = load_params(PROJECT_ROOT)
ACTIVATION_ID = params['activation_id']
COPERNICUS_FILE = copernicus_file(PROJECT_ROOT, ACTIVATION_ID)
print({'activation_id': ACTIVATION_ID, 'hazard_type': params.get('hazard_type'), 'copernicus_file': str(COPERNICUS_FILE)})

{'activation_id': 'EMSR884', 'hazard_type': 'earthquake', 'copernicus_file': 'c:\\Users\\loai\\Dropbox\\20251211-Action\\IFRC Project\\usecase2\\notebooks\\montandon_impact_estimation\\input\\copernicus\\EMSR884\\data.gpkg'}


## Load local Copernicus data

In [2]:
from src.copernicus_loader import filter_impact, layer_summary, load_copernicus_layers

copernicus_layers = load_copernicus_layers(COPERNICUS_FILE)
summary = layer_summary(copernicus_layers)
display(summary)

for name, layer in copernicus_layers.items():
    preview_cols = [c for c in layer.columns if c != 'geometry'][:8]
    print(name)
    display(layer[preview_cols].head())

,layer,feature_count,geometry_type,crs,columns
0,EMSR884_damage_confidence_20260701_v1,123633,MultiPolygon,EPSG:4326,"overture_id, damage_confidence, damage"


EMSR884_damage_confidence_20260701_v1


,overture_id,damage_confidence,damage
0,5f395eaa-f24b-439c-8dff-dbcbef0ff1c2,possible,0
1,9ea0fab7-a1f9-4480-944f-2b4d66f61ed8,possible,0
2,2d617c2a-d9f3-431a-a15d-759719d441b5,possible,0
3,c8a3016f-3023-4ff4-a693-769fe7be3e3f,high_confidence,1
4,4d8b030b-1aec-420a-b014-32f1cb6dfc9a,high_confidence,1


## Apply the damage filter used downstream
For EMSR884 the grading layer includes undamaged buildings (`damage = 0`). Notebook 02 keeps `damage = 1` only.

In [3]:
from src.copernicus_loader import concat_layers

impact_all = concat_layers(copernicus_layers)
impact = filter_impact(impact_all, params)
print({
    'features_loaded': len(impact_all),
    'features_after_damage_filter': len(impact),
    'bounds_wgs84': impact.to_crs('EPSG:4326').total_bounds.tolist() if not impact.empty else None,
})

{'features_loaded': 123633, 'features_after_damage_filter': 69431, 'bounds_wgs84': [-69.163528, 9.7933696, -66.200217, 11.0380141]}
